# 实践 7.1  RGB分割验证与动态表型计算

《统计遗传与智能育种学》第七章 · 电子资源 `07_phenomics_rgb.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：07_phenomics_rgb.ipynb。实践构建具有已知真值冠层比例的模拟RGB小区图像，在不同光照下使用归一化超绿指数（excess green index, ExG）进行绿色冠层分割，直接计算真值与估计覆盖度的相关和平均绝对误差；另行模拟多日期、三重复观测，明确注入一个预设异常重复并用组内中位数偏差识别，再在清理后计算AUC和早期生长速率。由于输入由代码生成，读者无需下载大型图像即可分别理解分割与真值验证、重复质量控制（quality control, QC）与动态性状计算两个环节。

关键输出：固定SEED=701时，模拟真值与分割估计覆盖度相关约1.00、平均绝对误差（mean absolute error, MAE）约0.003；成功识别1个预先注入的异常重复；输出清理后的AUC和早期生长速率。  
解释重点：这是受控合成数据上的算法单元测试，不代表真实田间分割精度。练习将ExG阈值由0.18改为0.12和0.24，说明本例AUC为何不随阈值改变。

## 输入、完整代码、实际输出与解释

输入与可复现条件：具有已知冠层比例的模拟RGB小区图像、多日期3重复观测及1个明确注入的异常重复；SEED=701；无需外部图像。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED=701
rng=np.random.default_rng(SEED)
print('Python',sys.version.split()[0])
print('Platform',platform.platform())
print('numpy',np.__version__,'pandas',pd.__version__,'scikit-learn',sklearn.__version__)
print('SEED',SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 701


In [2]:
# 生成具有已知真值冠层比例的RGB图像，并用归一化Excess Green分割
def make_plot(target_cover,light,L=64):
 yy,xx=np.mgrid[:L,:L]; cy=cx=(L-1)/2; radius=np.sqrt(target_cover*L*L/np.pi); mask=(xx-cx)**2+(yy-cy)**2<=radius**2
 img=np.empty((L,L,3),float); img[:]=[.35,.28,.18]; img[mask]=[.18,.55,.15]; img=np.clip(img*light+rng.normal(0,.015,img.shape),0,1)
 return img,float(mask.mean())
def canopy_fraction(img):
 s=img.sum(2)+1e-8; r,g,b=img[:,:,0]/s,img[:,:,1]/s,img[:,:,2]/s; exg=2*g-r-b; return float(np.mean(exg>.18))
truth=[]; estimated=[]
for cover in np.linspace(.10,.70,13):
 for light in [.7,1.0,1.3]:
  img,true_cover=make_plot(cover,light); truth.append(true_cover); estimated.append(canopy_fraction(img))
print('true-vs-estimated canopy r',round(np.corrcoef(truth,estimated)[0,1],3),'MAE',round(float(np.mean(np.abs(np.array(truth)-estimated))),4))

# 多日期重复观测：QC、AUC和早期生长速率
genos=[f'G{i:02d}' for i in range(24)]; days=np.array([10,18,26,34]); rows=[]
for gi,g in enumerate(genos):
 rate=rng.uniform(.015,.035); maxc=rng.uniform(.45,.82)
 for rep in range(3):
  for day in days:
   expected=maxc/(1+np.exp(-rate*8*(day-20))); obs=np.clip(expected+rng.normal(0,.025),0,1); rows.append((g,rep,day,obs))
df=pd.DataFrame(rows,columns=['genotype','rep','day','cover']); df.loc[(df.genotype=='G05')&(df.rep==2)&(df.day==26),'cover']+=.22 # 教学用异常重复
med=df.groupby(['genotype','day']).cover.transform('median'); df['rep_deviation']=np.abs(df.cover-med); flagged=df[df.rep_deviation>.12]; print('QC flagged replicate rows',len(flagged)); print(flagged[['genotype','rep','day','cover','rep_deviation']].round(3).to_string(index=False))
trait=[]
for g,sub in df.groupby('genotype'):
 clean=sub[sub.rep_deviation<=.12]; mean=clean.groupby('day').cover.mean().reindex(days).to_numpy(); auc=float(np.trapezoid(mean,days)); early=float((mean[1]-mean[0])/(days[1]-days[0])); trait.append((g,auc,early))
print(pd.DataFrame(trait,columns=['genotype','AUC','early_growth_rate']).head().round(4).to_string(index=False))


true-vs-estimated canopy r 1.0 MAE 0.0031
QC flagged replicate rows 1
genotype  rep  day  cover  rep_deviation
     G05    2   26   0.63          0.162
genotype     AUC  early_growth_rate
     G00  7.8498             0.0224
     G01  9.9291             0.0222
     G02  6.9017             0.0167
     G03 10.2555             0.0236
     G04  8.5192             0.0241


结果解释：ExG分割的真值—估计覆盖度相关1.000、MAE 0.0031，这是可控合成图像上的测量验证，不代表真实田间精度；QC准确标记1行预设异常后再计算AUC和早期生长速率。图像派生量必须经过测量验证和重复QC后才应进入遗传分析。

可修改练习：把ExG阈值0.18改为0.12和0.24，比较覆盖度MAE并说明AUC为何不变；再加入不均匀阴影观察固定阈值的失效。